# Huấn Luyện Mô Hình MulCo (End-to-End)
Notebook này cho phép bạn cấu hình môi trường, lắp ráp kiến trúc từ các module độc lập (Fusion, Classifier) và tiến hành huấn luyện.

In [ ]:
# Gỡ bỏ sạch sẽ các tàn dư của phiên bản cũ
!pip uninstall -y torch torchvision torchaudio

# Cài đặt lại mới hoàn toàn
!pip install torch torchvision torchaudio transformers


In [1]:
import os
import sys
from pathlib import Path

# Thêm thư mục gốc vào sys.path để import các module từ src
current_dir = Path.cwd()
PROJECT_ROOT = current_dir
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))
print(f"Project Root: {PROJECT_ROOT}")

Project Root: /media/data3/users/luongdth/MulCo-PlantNet


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from transformers import CLIPTokenizer, CLIPModel
from tqdm.notebook import tqdm

from src.datasets.mulco_dataset import MulCoDataset
from src.models.convnext_cbam import ConvNeXt_CBAM
from src.models.mulco_fusion import MulCoFusionBlock
from src.models.mulco_classifier import Conv1x1Classifier

ImportError: /media/data3/users/luongdth/anaconda3/envs/gr1/lib/python3.12/site-packages/torch/lib/libtorch_cuda.so: undefined symbol: ncclCommWindowDeregister

## 1. Lắp Ráp Kiến Trúc (Model Assembly)

In [ ]:
class MulCoEndToEnd(nn.Module):
    def __init__(self, num_classes=28, proj_dim=512, spatial_size=(7, 7)):
        super().__init__()
        self.image_backbone = ConvNeXt_CBAM(num_classes=num_classes)
        self.text_backbone = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").text_model
        
        self.img_proj = nn.Conv2d(1024, proj_dim, kernel_size=1)
        self.txt_proj = nn.Linear(512, proj_dim)
        self.dropout = nn.Dropout(0.3)
        
        self.fusion_blocks = nn.ModuleList([
            MulCoFusionBlock(dim=proj_dim, num_heads=8) for _ in range(3)
        ])
        
        self.classifier = Conv1x1Classifier(in_channels=proj_dim, num_classes=num_classes, spatial_size=spatial_size)

    def forward(self, images, input_ids, attention_mask):
        img_feat = self.image_backbone.forward_features_spatial(images) 
        txt_out = self.text_backbone(input_ids=input_ids, attention_mask=attention_mask)
        txt_feat = txt_out.last_hidden_state
        
        img_feat = self.img_proj(img_feat)
        txt_feat = self.txt_proj(txt_feat)
        img_feat = self.dropout(img_feat)
        txt_feat = self.dropout(txt_feat)
        
        for block in self.fusion_blocks:
            img_feat, txt_feat = block(img_feat, txt_feat)
            
        logits = self.classifier(img_feat)
        return logits

## 2. Tiền Xử Lý Dữ Liệu & Khởi Tạo Vòng Lặp Huấn Luyện

In [ ]:
def custom_collate_fn(batch, tokenizer):
    images = torch.stack([b["image"] for b in batch])
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    texts = [b["text"] for b in batch]
    text_tokens = tokenizer(texts, padding=True, truncation=True, max_length=77, return_tensors="pt")
    return images, text_tokens.input_ids, text_tokens.attention_mask, labels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1))
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

train_dataset = MulCoDataset(
    image_root=os.path.join(PROJECT_ROOT, "data/AIDG/dataset_PlantDoc/images/train"),
    caption_root=os.path.join(PROJECT_ROOT, "data/AIDG/captions_LLaVA/train"),
    transform=train_transform,
    use_depth_suppressed=False,
    strict_caption_match=False
)

train_loader = DataLoader(
    train_dataset, batch_size=16, shuffle=True, num_workers=2,
    collate_fn=lambda b: custom_collate_fn(b, tokenizer)
)

val_dataset = MulCoDataset(
    image_root=os.path.join(PROJECT_ROOT, "data/AIDG/dataset_PlantDoc/images/val"),
    caption_root=os.path.join(PROJECT_ROOT, "data/AIDG/captions_LLaVA/val"),
    transform=val_transform,
    use_depth_suppressed=False,
    strict_caption_match=False
)

val_loader = DataLoader(
    val_dataset, batch_size=16, shuffle=False, num_workers=2,
    collate_fn=lambda b: custom_collate_fn(b, tokenizer)
)

model = MulCoEndToEnd(num_classes=28).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=5e-2)

epochs = 10
best_val_acc = 0.0
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = os.path.join(PROJECT_ROOT, "archive", "cross_attention_3blocks")
os.makedirs(save_dir, exist_ok=True)
print(f"Models will be saved to: {save_dir}")

for epoch in range(epochs):
    model.train()
    total_loss, correct, total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    
    for images, input_ids, attn_mask, labels in pbar:
        images, input_ids, attn_mask, labels = images.to(device), input_ids.to(device), attn_mask.to(device), labels.to(device)
        
        optimizer.zero_grad()
        logits = model(images, input_ids, attn_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix({"Loss": total_loss/total, "Acc": correct/total})
        
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, input_ids, attn_mask, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False):
            images, input_ids, attn_mask, labels = images.to(device), input_ids.to(device), attn_mask.to(device), labels.to(device)
            logits = model(images, input_ids, attn_mask)
            preds = torch.argmax(logits, dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    
    val_acc = val_correct / val_total
    print(f"Epoch {epoch+1} - Train Acc: {correct/total:.4f}, Val Acc: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), os.path.join(save_dir, "best_model.pth"))
        print(f"--> Đã lưu Best Model (Val Acc: {best_val_acc:.4f})")

torch.save(model.state_dict(), os.path.join(save_dir, "last_model.pth"))
print("Hoàn tất huấn luyện! Đã lưu Last Model.")
